# Fundamentals 07 - Integrations OpenAI Agents API

**Historia:** Agentic Systems define el contrato; `openai-agents` declara el framework y `openai-runtime` queda como provider/engine nativo que ejecuta el loop con el SDK oficial de OpenAI.

La fachada sigue empezando en Agentic Systems:

```text
runtime = toolkit.runtime(provider="auto")
toolkit.agent(..., runtime=runtime, framework="openai-agents") -> await agent.arun(...)
```

Nota de arquitectura: este notebook usa `framework="openai-agents"` como fachada de integracion. El backend sigue siendo agnostico y lo decide `runtime(provider="auto")`; cuando resuelve OpenAI, usa `openai-runtime` como provider nativo.

In [ ]:
import agentic_systems as toolkit

PRETTY = False  # Cambia a True para usar Rich; False imprime texto plano estable y reproducible.

scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=4, max_turns=8)
runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
runtime_description = runtime.describe()
resolved_provider = runtime_description.get("selected_provider")

toolkit.show({
    "runtime_auto_resolution": runtime_description,
    "resolved_provider": resolved_provider,
    "human_results_pretty": PRETTY,
})

## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Definir tools explicitas


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos numeros."""
    return {"operation": "sumar", "result": a + b}

@toolkit.tool
def restar(a: int, b: int) -> dict:
    """Resta dos numeros."""
    return {"operation": "restar", "result": a - b}

@toolkit.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos numeros."""
    return {"operation": "multiplicar", "result": a * b}

@toolkit.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos numeros."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    return {"operation": "dividir", "result": int(value) if value.is_integer() else value}

tools = [sumar, restar, multiplicar, dividir]

toolkit.show({
    "tools": [tool.name for tool in tools],
    "mental_model": "tools explicitas  agent = toolkit.agent(..., tools=tools)",
})


## 2) Crear agente portable con framework OpenAI Agents

Aqui se ven separadas las dos decisiones:

```text
runtime   = backend de inferencia/modelo seleccionado por provider="auto"
framework = loop/adaptador de agente
```

Primero se declara `runtime`. Despues `framework="openai-agents"` obedece ese runtime: no fuerza Bedrock, no fuerza OpenAI si `auto` no lo resolvio, y mantiene separada la fachada del provider.

In [ ]:
instructions = """
Eres un agente calculadora.
Usa las tools disponibles para resolver la solicitud con evidencia auditable.
"""

# Importante: `framework` no decide provider. Congelamos el provider resuelto por
# runtime(provider="auto") y lo pasamos como engine para evitar una segunda
# resolucion `auto` que pueda caer en Bedrock si existe region AWS sin credenciales.
engine = resolved_provider
INTEGRATION_READY = False
if not engine or engine == "auto":
    toolkit.show({
        "status": "skipped",
        "reason": "provider='auto' no resolvio un engine concreto para la integracion",
        "auto_resolution": runtime_description,
    }, title="Integracion saltada")
else:
    system = toolkit.AgenticSystem(
        model=runtime.model_id,
        region=runtime.region_name if engine == "bedrock-runtime" else None,
        runtime=runtime,
    )

    agent = system.agent(
        name="calculator_openai_agents_integration",
        instructions=instructions,
        tools=tools,
        engine=engine,
        runtime=runtime,
        framework="openai-agents",
    )

    INTEGRATION_READY = True

    toolkit.show({
        "agent_name": agent.name,
        "engine": agent.engine,
        "framework": agent.framework,
        "resolved_provider": engine,
        "auto_resolution": runtime_description,
        "tools": [tool.name for tool in tools],
        "what_is_hidden": "nada de negocio: solo el adapter del loop externo",
        "what_is_still_agentic_systems": ["runtime", "tools", "contract/result envelope", "human_result", "lineage"],
    })


## 3) Ejecutar de forma asincrona

La integracion se prueba con `await agent.arun(...)` para ensenar la ruta asincrona nativa. Si `provider="auto"` no encontro backend configurado, la celda salta la ejecucion en vez de fingir que hay OpenAI o Bedrock disponible.

In [ ]:
user_prompt = USER_PROMPT

if not INTEGRATION_READY:
    result = None
    toolkit.show({
        "status": "skipped",
        "reason": runtime_description.get("reason"),
        "how_to_enable": "Configura OPENAI_API_KEY o credenciales/configuracion Bedrock antes de abrir el kernel.",
    }, title="OpenAI Runtime integration saltada")
else:
    result = await agent.arun(user_prompt, mode="eval")

    toolkit.human_result(
        result,
        title="Human result - OpenAI Agents framework integration",
        expected_tools=toolkit.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        pretty=PRETTY,
    )

## 4) Definir `finalize`: evidencia  salidas solicitadas

`finalize` no es otra tool y no llama LLM. Es el cierre del loop: toma la evidencia estructurada de las tools y rellena lo que el usuario pidio (`procedimiento`, `resultado_final`).

La respuesta libre del agente puede ser util como narracion, pero la salida final auditable se arma desde los tool outputs.

In [ ]:
def _tool_data(tool: dict) -> dict:
    output = tool.get("output") if isinstance(tool, dict) else {}
    if isinstance(output, dict) and isinstance(output.get("data"), dict):
        return output["data"]
    return output if isinstance(output, dict) else {}


def _procedure_from_tools(tools: list[dict]) -> list[str]:
    symbols = {"sumar": "+", "restar": "-", "multiplicar": "", "dividir": ""}
    lines = []
    for tool in tools:
        name = tool.get("name")
        payload = _tool_data(tool)
        tool_input = tool.get("input") if isinstance(tool.get("input"), dict) else {}
        result_value = payload.get("result", payload.get("value"))
        if name in symbols and {"a", "b"}.issubset(tool_input) and result_value is not None:
            lines.append(f"{tool_input['a']} {symbols[name]} {tool_input['b']} = {result_value}")
    return lines


def finalize(run_result, requested_outputs: list[str]) -> dict:
    normalized = run_result.normalized() if hasattr(run_result, "normalized") else {}
    tools = normalized.get("tools") if isinstance(normalized.get("tools"), list) else []
    procedure = _procedure_from_tools(tools)

    final_value = None
    for tool in reversed(tools):
        data = _tool_data(tool)
        final_value = data.get("result", data.get("value"))
        if final_value is not None:
            break

    requested = {}
    if "procedimiento" in requested_outputs:
        requested["procedimiento"] = procedure
    if "resultado_final" in requested_outputs:
        requested["resultado_final"] = final_value

    return {
        "requested_outputs": requested_outputs,
        "final_answer": requested,
        "source": "structured_tool_outputs",
        "agent_text_observation": (normalized.get("answer") or {}).get("text"),
    }


finalized = finalize(result, REQUESTED_OUTPUTS)
toolkit.show(finalized, title="Finalize  evidencia  salidas solicitadas")

## 5) Lineage Memory transversal del loop externo

El resultado sigue siendo `RunResult`, pero el adapter permite proyectar una traza de Strands o un resultado envuelto por Agentic Systems al mismo formato de explicacion.

In [ ]:
if result is None:
    toolkit.show({"status": "skipped", "reason": "no hay resultado LM para lineage"}, title="Lineage Memory saltado")
else:
    lineage = result.lineage(
        name="fundamentals.calculator.openai_agents",
        question=user_prompt,
        goal="Explicar un loop OpenAI Runtime con la misma API de Lineage Memory.",
    )

    toolkit.show(lineage, title="Lineage Memory ? OpenAI Runtime")

    toolkit.human_result(
        result,
        title="Human result + Lineage Memory ? OpenAI Runtime",
        expected_tools=toolkit.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        pretty=PRETTY,
        show_lineage=True,
        lineage=lineage,
    )

## Lo importante

- Agentic Systems sigue siendo la fachada publica.
- `runtime(provider="auto")` selecciona backend por entorno; el notebook no fuerza Bedrock ni OpenAI.
- `framework="openai-agents"` declara la fachada/framework, no el provider.
- Este notebook declara `framework="openai-agents"`; el provider ejecutable sigue siendo el runtime seleccionado por `provider="auto"`.
- `finalize(...)` materializa las salidas solicitadas desde evidencia estructurada, no desde texto libre.
- El resultado sigue siendo explicable con `toolkit.human_result`.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": 'runtime(provider="auto")',
        "description": "Selecciona backend por configuracion del ambiente sin acoplar el tutorial."
    },
    {
        "api": 'framework="openai-agents"',
        "description": "Declara la fachada OpenAI Agents sin acoplar el provider ejecutable."
    },
    {
        "api": "agent.arun",
        "description": "Demuestra ejecucion asincrona del agente con el framework elegido."
    },
    {
        "api": "finalize(result, requested_outputs)",
        "description": "Cierra la salida con una funcion explicita y auditable."
    },
    {
        "api": "RunResult.lineage",
        "description": "Proyecta cualquier RunResult canonico a Lineage Memory sin helpers legacy."
    },
    {
        "api": "human_result(show_lineage=True)",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "default arithmetic prompt",
        "description": "Usa el mismo prompt base para comparar comportamiento entre notebooks."
    }
]

toolkit.show({'notebook': '07_integrations_openai_runtime_api.ipynb', 'api_coverage': api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `framework="openai-agents"`: Integracion OpenAI Agents como framework.
- `openai-runtime`: Engine nativo que puede cruzarse con frameworks.
- `AgenticSystem`: Fachada nativa agnostica.
- `provider="auto"`: Seleccion automatica del provider.
- `human_result`: Render de la ejecucion de integracion.

